# Database-Connected Agents — SQLite Only

This notebook is the SQLite-only version of the database-connected agents lab.

What it covers:
- a local serverless SQLite database
- agent tools for schema inspection and read-only querying
- Groq for the agent LLM
- structured output from the agent into database rows
- a simple model split for different agent tasks

LangChain’s SQL-agent guide shows the core workflow: inspect tables and schemas, decide what is relevant, generate a query, execute it, and handle SQL mistakes carefully. The tools docs show the basic `@tool` pattern, and the structured-output docs show that `create_agent` can return validated data in `structured_response`.


## Learning goals

By the end of this notebook, you should be able to:

1. Work with a local SQLite database only.
2. Build tools for table listing, schema inspection, and read-only querying.
3. Use Groq as the agent model.
4. Return structured output from the agent.
5. Save the structured result back into SQLite.
6. Use different models for SQL reasoning and row extraction.


## 1) Install packages


In [ ]:
%pip install -qU \
    python-dotenv requests \
    langchain langchain-core langchain-groq \
    sqlalchemy pydantic


## 2) Environment variables

Create a `.env` file with the values you actually have:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=database-connected-agents-sqlite
```

LangChain’s structured-output and agents docs show that `create_agent` can use `response_format` to return typed structured data. 


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
LANGSMITH_TRACING = os.getenv('LANGSMITH_TRACING', 'true').lower() == 'true'
LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', 'database-connected-agents-sqlite')

print('GROQ_API_KEY set:', bool(GROQ_API_KEY))
print('LANGSMITH_API_KEY set:', bool(LANGSMITH_API_KEY))
print('LANGSMITH_TRACING:', LANGSMITH_TRACING)
print('LANGSMITH_PROJECT:', LANGSMITH_PROJECT)


GROQ_API_KEY set: True
LANGSMITH_API_KEY set: True
LANGSMITH_TRACING: True
LANGSMITH_PROJECT: lcel-groq-demo


## 3) Create a local SQLite database

To keep this fully serverless and local, we use a single SQLite file on disk.

The database is a small e-commerce style sample with customers, orders, order items, and support tickets.


In [2]:
import sqlite3
from pathlib import Path
from datetime import datetime

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

APP_DB = DATA_DIR / 'app_local.db'

if APP_DB.exists():
    APP_DB.unlink()

conn = sqlite3.connect(APP_DB)
cur = conn.cursor()

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_name TEXT NOT NULL,
    city TEXT NOT NULL,
    signup_date TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    total_amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

cur.execute("""
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)
""")

cur.execute("""
CREATE TABLE support_tickets (
    ticket_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    subject TEXT NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)
""")

customers = [
    ('Asha Rao', 'Chennai', '2026-01-02'),
    ('Imran Khan', 'Bengaluru', '2026-01-10'),
    ('Neha Patel', 'Mumbai', '2026-01-18'),
]
cur.executemany(
    'INSERT INTO customers (full_name, city, signup_date) VALUES (?, ?, ?)',
    customers
)

orders = [
    (1, '2026-02-01', 'completed', 2499.0),
    (1, '2026-02-12', 'completed', 899.0),
    (2, '2026-02-14', 'processing', 1499.0),
    (3, '2026-02-20', 'completed', 3199.0),
]
cur.executemany(
    'INSERT INTO orders (customer_id, order_date, status, total_amount) VALUES (?, ?, ?, ?)',
    orders
)

items = [
    (1, 'Keyboard', 1, 2499.0),
    (2, 'Mouse', 1, 899.0),
    (3, 'Headphones', 1, 1499.0),
    (4, 'Monitor', 1, 3199.0),
]
cur.executemany(
    'INSERT INTO order_items (order_id, product_name, quantity, unit_price) VALUES (?, ?, ?, ?)',
    items
)

tickets = [
    (1, 'Refund request', 'open', '2026-02-03T09:00:00'),
    (2, 'Order status question', 'closed', '2026-02-15T10:30:00'),
    (3, 'Invoice needed', 'open', '2026-02-21T08:45:00'),
]
cur.executemany(
    'INSERT INTO support_tickets (customer_id, subject, status, created_at) VALUES (?, ?, ?, ?)',
    tickets
)

conn.commit()
conn.close()

print('Created local SQLite database:', APP_DB.resolve())


Created local SQLite database: D:\personal_docs\course-ai\module-1\agenticai-with-langgraph\data\app_local.db


## 4) Inspect the database


In [3]:
conn = sqlite3.connect(APP_DB)
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]

print('Tables:', tables)

for table in tables:
    cur.execute(f'PRAGMA table_info({table})')
    cols = cur.fetchall()
    print(f'\nSchema for {table}')
    for c in cols:
        print(f'- {c[1]} ({c[2]})')

conn.close()


Tables: ['customers', 'order_items', 'orders', 'sqlite_sequence', 'support_tickets']

Schema for customers
- customer_id (INTEGER)
- full_name (TEXT)
- city (TEXT)
- signup_date (TEXT)

Schema for order_items
- item_id (INTEGER)
- order_id (INTEGER)
- product_name (TEXT)
- quantity (INTEGER)
- unit_price (REAL)

Schema for orders
- order_id (INTEGER)
- customer_id (INTEGER)
- order_date (TEXT)
- status (TEXT)
- total_amount (REAL)

Schema for sqlite_sequence
- name ()
- seq ()

Schema for support_tickets
- ticket_id (INTEGER)
- customer_id (INTEGER)
- subject (TEXT)
- status (TEXT)
- created_at (TEXT)


## 5) Define SQLite tools

The tools docs show the basic `@tool` pattern, and LangChain agents can call those tools during the reasoning loop. 


In [4]:
from langchain.tools import tool

@tool
def sqlite_list_tables() -> str:
    """List tables in the local SQLite database."""
    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    rows = [r[0] for r in cur.fetchall() if not r[0].startswith('sqlite_')]
    conn.close()
    return ', '.join(rows)

@tool
def sqlite_schema(table_name: str) -> str:
    """Return the schema for a SQLite table."""
    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    cur.execute(f'PRAGMA table_info({table_name})')
    cols = cur.fetchall()
    conn.close()
    if not cols:
        return f'No such table: {table_name}'
    return '\n'.join([f'{c[1]} ({c[2]})' for c in cols])

@tool
def sqlite_query(query: str) -> str:
    """Run a read-only SELECT/WITH query against SQLite."""
    q = query.strip().lower()
    forbidden = ['insert ', 'update ', 'delete ', 'drop ', 'alter ', 'create ']
    if not q.startswith('select') and not q.startswith('with'):
        return 'Only read-only SELECT/WITH queries are allowed.'
    if any(word in q for word in forbidden):
        return 'Potentially unsafe SQL was blocked.'

    conn = sqlite3.connect(str(APP_DB))
    cur = conn.cursor()
    try:
        cur.execute(query)
        rows = cur.fetchall()
        return str(rows)
    except Exception as e:
        return f'SQLite error: {e}'
    finally:
        conn.close()

print(sqlite_list_tables.invoke({}))


C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


customers, order_items, orders, support_tickets


## 6) Choose models per task

A practical split is:
- one model for SQL reasoning
- one smaller model for extraction and formatting

This notebook keeps the split simple so you can change it later without rewriting the workflow.


In [5]:
from langchain_groq import ChatGroq

MODEL_REGISTRY = {
    'sql': os.getenv('GROQ_SQL_MODEL', 'llama-3.3-70b-versatile'),
    'extract': os.getenv('GROQ_EXTRACT_MODEL', 'llama-3.1-8b-instant'),
}

def build_model(node_name: str):
    return ChatGroq(
        model=MODEL_REGISTRY[node_name],
        temperature=0,
    )

sql_model = build_model('sql')
extract_model = build_model('extract')

print(MODEL_REGISTRY)


{'sql': 'llama-3.3-70b-versatile', 'extract': 'llama-3.1-8b-instant'}


## 7) Define structured output schemas

The structured-output docs say agents can return a validated `structured_response`. A Pydantic model is a convenient schema for that. 


In [6]:
from pydantic import BaseModel, Field

class QueryResult(BaseModel):
    question: str = Field(..., description='Original user question')
    database_name: str = Field(..., description='sqlite')
    sql_query: str = Field(..., description='The SQL query that was used')
    answer: str = Field(..., description='Short answer to the user')
    row_count: int = Field(..., description='How many rows were returned')

class AgentInsight(BaseModel):
    question: str = Field(..., description='Original user question')
    database_name: str = Field(..., description='sqlite')
    summary: str = Field(..., description='Short operational summary')
    sql_query: str = Field(..., description='The query that was executed')
    row_count: int = Field(..., description='Row count from the query')


## 8) Build the SQLite agent

The SQL-agent tutorial describes a loop where the agent inspects tables, checks schemas, builds a query, and executes it with care. Here we keep that pattern but only for SQLite.


In [7]:
from langchain.agents import create_agent

sql_agent = create_agent(
    sql_model,
    tools=[sqlite_list_tables, sqlite_schema, sqlite_query],
    system_prompt=(
        'You are a SQLite database-connected agent. '
        'Use the SQLite tools to inspect the schema and answer questions. '
        'Always prefer a read-only query. '
        'Keep SQL small and focused.'
    ),
    response_format=QueryResult,
)

print('SQLite agent ready.')


SQLite agent ready.


## 9) Ask a SQLite question


In [9]:
question_1 = 'How many tracks are in the local database?'

result_1 = sql_agent.invoke({
    'messages': [{'role': 'user', 'content': question_1}]
})

result_1


{'messages': [HumanMessage(content='How many tracks are in the local database?', additional_kwargs={}, response_metadata={}, id='5120a082-be1b-4996-95d8-6241af757afc'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3bzz2hezp', 'function': {'arguments': '{"query":"SELECT COUNT(*) FROM tracks;"}', 'name': 'sqlite_query'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 489, 'total_tokens': 505, 'completion_time': 0.064397829, 'completion_tokens_details': None, 'prompt_time': 0.100121043, 'prompt_tokens_details': None, 'queue_time': 0.052035635, 'total_time': 0.164518872}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebbbb-d36d-70a0-9bf0-9e0b622a0e95-0', tool_calls=[{'name': 'sqlite_query', 'args': {'query': 'SELECT COUNT(*) FROM tracks;'}, 'id': '3bzz2hezp', 'type'

## 10) Inspect the structured response


In [10]:
structured_1 = result_1['structured_response']
structured_1


QueryResult(question='How many tracks are in the local database?', database_name='sqlite', sql_query='SELECT COUNT(*) FROM orders;', answer='4', row_count=1)

## 11) Save the structured response back into SQLite

This is the structured-output-to-database pattern.

We insert the model-generated row into the local `agent_insights` table.


In [12]:
# Create audit database

AUDIT_DB = DATA_DIR / "agent_audit.db"

audit_conn = sqlite3.connect(AUDIT_DB)
audit_cur = audit_conn.cursor()

audit_cur.execute("""
CREATE TABLE IF NOT EXISTS agent_insights (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    database_name TEXT NOT NULL,
    sql_query TEXT,
    answer TEXT NOT NULL,
    row_count INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
""")

audit_conn.commit()
audit_conn.close()

print("Audit DB ready:", AUDIT_DB)

Audit DB ready: data\agent_audit.db


In [13]:
audit_conn = sqlite3.connect(str(AUDIT_DB))
audit_cur = audit_conn.cursor()

audit_cur.execute(
    'INSERT INTO agent_insights (question, database_name, sql_query, answer, row_count, created_at) VALUES (?, ?, ?, ?, ?, ?)',
    (
        structured_1.question,
        structured_1.database_name,
        structured_1.sql_query,
        structured_1.answer,
        structured_1.row_count,
        datetime.utcnow().isoformat(),
    ),
)

audit_conn.commit()
audit_cur.execute('SELECT question, database_name, answer, row_count FROM agent_insights')
saved_rows = audit_cur.fetchall()
audit_conn.close()

saved_rows


C:\Users\MadhiarasanM\AppData\Local\Temp\ipykernel_25292\266703073.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().isoformat(),


[('How many tracks are in the local database?', 'sqlite', '4', 1)]

## 12) Build a smaller extraction agent

The smaller model is useful for cleanup and row formatting rather than SQL reasoning.


In [14]:
extract_agent = create_agent(
    extract_model,
    tools=[],
    response_format=AgentInsight,
)

print('Extraction agent ready.')


Extraction agent ready.


In [15]:
insight_result = extract_agent.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            'Convert this SQLite agent result into a clean row: '
            f'question={structured_1.question}; '
            f'database={structured_1.database_name}; '
            f'sql={structured_1.sql_query}; '
            f'answer={structured_1.answer}; '
            f'row_count={structured_1.row_count}'
        )
    }]
})

insight_result['structured_response']


AgentInsight(question='How many tracks are in the local database?', database_name='sqlite', summary='Executed query and returned 1 row(s)', sql_query='SELECT COUNT(*) FROM orders', row_count=1)

## 13) Model choice by node

Use a stronger model for SQL planning and a smaller model for extraction. That keeps the notebook practical and inexpensive while still showing the pattern clearly.


In [16]:
def choose_node_model(node_name: str) -> str:
    return MODEL_REGISTRY[node_name]

print('sql node model:', choose_node_model('sql'))
print('extract node model:', choose_node_model('extract'))


sql node model: llama-3.3-70b-versatile
extract node model: llama-3.1-8b-instant


## 14) A simple SQLite-only router

You can keep routing basic and entirely local.


In [17]:
def choose_database(question: str) -> str:
    return 'sqlite'

for q in [
    'How many tracks are there?',
    'What is the refund policy?',
]:
    print(q, '->', choose_database(q))


How many tracks are there? -> sqlite
What is the refund policy? -> sqlite


## 15) Why this version is better

This version is simpler and more useful because it:
- uses only SQLite
- runs locally as a single serverless file
- avoids PostgreSQL configuration entirely
- still demonstrates tools, structured output, and database writes


## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Structured output: https://docs.langchain.com/oss/python/langchain/structured-output
- Tools: https://docs.langchain.com/oss/python/langchain/tools
